# Task 1: Business questions and data cleaning

## Business questions

1. Which bookings are likely to be cancelled, and which guests should receive a reminder or deposit request?
2. What is a reasonable expected ADR for a booking at the time it is created?
3. When is recorded hotel demand highest, and which booking characteristics are associated with that demand?

The source does not contain historical room inventory, so it cannot prove that a hotel was fully occupied. The third question therefore measures recorded demand rather than occupancy.


In [1]:
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
np.random.seed(SEED)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid')

DATA_DIR = Path('../data')
OUTPUT_DIR = Path('../outputs')
SLIDE_DIR = Path('../slides')
MODEL_DIR = Path('../models')
for folder in [DATA_DIR, OUTPUT_DIR, SLIDE_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


## Load and profile the original data

The original CSV remains unchanged. A stable `source_row_id` is added so every cleaned row can be traced back to the source file.


In [2]:
raw = pd.read_csv(DATA_DIR / 'hotel_bookings.csv')
raw.insert(0, 'source_row_id', np.arange(len(raw), dtype='int64'))
print('Raw shape:', raw.shape)

profile = []
for col in raw.columns:
    s = raw[col]
    profile.append({
        'column': col,
        'dtype': str(s.dtype),
        'missing': int(s.isna().sum()),
        'missing_pct': round(float(s.isna().mean() * 100), 2),
        'unique': int(s.nunique(dropna=True)),
        'min': s.min() if pd.api.types.is_numeric_dtype(s) else (s.dropna().astype(str).min() if s.notna().any() else None),
        'max': s.max() if pd.api.types.is_numeric_dtype(s) else (s.dropna().astype(str).max() if s.notna().any() else None),
    })
profile_df = pd.DataFrame(profile)
display(profile_df)
profile_df.to_csv(OUTPUT_DIR / 'column_profile.csv', index=False)


Raw shape: (119390, 33)
                            column    dtype  ...         min              max
0                    source_row_id    int64  ...           0           119389
1                            hotel   object  ...  City Hotel     Resort Hotel
2                      is_canceled    int64  ...           0                1
3                        lead_time    int64  ...           0              737
4                arrival_date_year    int64  ...        2015             2017
5               arrival_date_month   object  ...       April        September
6         arrival_date_week_number    int64  ...           1               53
7        arrival_date_day_of_month    int64  ...           1               31
8          stays_in_weekend_nights    int64  ...           0               19
9             stays_in_week_nights    int64  ...           0               50
10                          adults    int64  ...           0               55
11                        children  floa

                            column    dtype  ...         min              max
0                    source_row_id    int64  ...           0           119389
1                            hotel   object  ...  City Hotel     Resort Hotel
2                      is_canceled    int64  ...           0                1
3                        lead_time    int64  ...           0              737
4                arrival_date_year    int64  ...        2015             2017
5               arrival_date_month   object  ...       April        September
6         arrival_date_week_number    int64  ...           1               53
7        arrival_date_day_of_month    int64  ...           1               31
8          stays_in_weekend_nights    int64  ...           0               19
9             stays_in_week_nights    int64  ...           0               50
10                          adults    int64  ...           0               55
11                        children  float64  ...         0.0    

## Quality checks and cleaning decisions

Exact duplicate rows are flagged but retained because the source has no booking ID. Identical rows may represent separate group bookings. Rows with no guests and the one negative ADR record are removed. Zero-night records and extreme values remain available with review flags.

The source target `is_canceled` is preserved. In this dataset, both `Canceled` and `No-Show` records have `is_canceled = 1`; changing No-Shows to completed stays would understate cancellation and overstate revenue.


In [3]:
df = raw.copy()
original_columns = [c for c in df.columns if c != 'source_row_id']
df['exact_duplicate_flag'] = df.duplicated(subset=original_columns, keep=False).astype('int8')

children_zero = df['children'].fillna(0).eq(0)
no_guest_flag = df['adults'].eq(0) & children_zero & df['babies'].eq(0)
zero_night_flag = df['stays_in_weekend_nights'].eq(0) & df['stays_in_week_nights'].eq(0)
negative_adr_flag = df['adr'].lt(0)

issue_counts = {
    'Exact duplicate rows retained and flagged': int(df['exact_duplicate_flag'].sum()),
    'Bookings with no guests removed': int(no_guest_flag.sum()),
    'Zero-night bookings retained and flagged': int(zero_night_flag.sum()),
    'Negative ADR rows removed': int(negative_adr_flag.sum()),
    'ADR above EUR 1000 retained and flagged': int(df['adr'].gt(1000).sum()),
    'Adults above 10 retained and flagged': int(df['adults'].gt(10).sum()),
    'Missing children filled with zero': int(df['children'].isna().sum()),
    'Missing country filled with UNK and flagged': int(df['country'].isna().sum()),
    'Missing agent retained and flagged': int(df['agent'].isna().sum()),
    'Missing company retained and flagged': int(df['company'].isna().sum()),
}

df = df.loc[~no_guest_flag & ~negative_adr_flag].copy()
df['zero_night_flag'] = (df['stays_in_weekend_nights'].eq(0) & df['stays_in_week_nights'].eq(0)).astype('int8')
df['adr_extreme_flag'] = df['adr'].gt(1000).astype('int8')
df['adults_extreme_flag'] = df['adults'].gt(10).astype('int8')
df['children_missing_flag'] = df['children'].isna().astype('int8')
df['country_missing_flag'] = df['country'].isna().astype('int8')
df['agent_missing_flag'] = df['agent'].isna().astype('int8')
df['company_missing_flag'] = df['company'].isna().astype('int8')

df['children'] = df['children'].fillna(0).astype('int16')
df['country'] = df['country'].fillna('UNK')
df['meal'] = df['meal'].replace({'Undefined': 'SC'})
df['reservation_status_date'] = pd.to_datetime(df['reservation_status_date'], errors='coerce')

month_map = {m: i for i, m in enumerate([
    'January','February','March','April','May','June',
    'July','August','September','October','November','December'
], start=1)}
df['arrival_date_month_num'] = df['arrival_date_month'].map(month_map).astype('int8')
df['arrival_date'] = pd.to_datetime(dict(
    year=df['arrival_date_year'],
    month=df['arrival_date_month_num'],
    day=df['arrival_date_day_of_month'],
), errors='coerce')
df['arrival_date_invalid_flag'] = df['arrival_date'].isna().astype('int8')
df['no_show_flag'] = df['reservation_status'].eq('No-Show').astype('int8')

expected_target = df['reservation_status'].isin(['Canceled', 'No-Show']).astype('int8')
target_mismatch = int((df['is_canceled'].astype('int8') != expected_target).sum())
issue_counts['Target/status mismatches after correct No-Show mapping'] = target_mismatch

assert target_mismatch == 0
assert df['arrival_date'].notna().all()
assert (df['adr'] >= 0).all()
print('Clean shape:', df.shape)
print('Cancellation rate:', f"{df['is_canceled'].mean():.1%}")
print('No-Show rows:', int(df['no_show_flag'].sum()))


Clean shape: (119209, 45)
Cancellation rate: 37.1%
No-Show rows: 1206


## Data report and outputs

Counts below use the original source population unless the description explicitly says otherwise. Overlapping issues must not be added together.


In [4]:
decisions = {
    'Exact duplicate rows retained and flagged': ('Keep and flag', 'No booking ID exists, so identical rows cannot be proven to be duplicate events.'),
    'Bookings with no guests removed': ('Remove', 'A booking requires at least one guest.'),
    'Zero-night bookings retained and flagged': ('Keep and flag', 'Some may be day-use records or data errors; the source does not distinguish them.'),
    'Negative ADR rows removed': ('Remove', 'A negative quoted room price is not valid for this analysis.'),
    'ADR above EUR 1000 retained and flagged': ('Keep and flag', 'The record may be valid but is influential for averages and models.'),
    'Adults above 10 retained and flagged': ('Keep and flag', 'The row may be a group booking.'),
    'Missing children filled with zero': ('Fill with zero', 'Four missing values have no other invalid guest information.'),
    'Missing country filled with UNK and flagged': ('Fill category and flag', 'The booking remains usable while missingness stays visible.'),
    'Missing agent retained and flagged': ('Keep missing and flag', 'Missing does not reliably prove a direct booking.'),
    'Missing company retained and flagged': ('Keep missing and flag', 'Missing does not reliably prove a non-corporate booking.'),
    'Target/status mismatches after correct No-Show mapping': ('No overwrite', 'The original target already treats Canceled and No-Show as cancellation outcomes.'),
}
data_report = pd.DataFrame([
    {'problem': name, 'rows': count, 'decision': decisions[name][0], 'reason': decisions[name][1]}
    for name, count in issue_counts.items()
])
display(data_report)

df.to_csv(DATA_DIR / 'hotel_bookings_clean.csv', index=False)
data_report.to_csv(OUTPUT_DIR / 'data_quality_report.csv', index=False)
print('Saved clean data and quality report.')


                                              problem  ...                                             reason
0           Exact duplicate rows retained and flagged  ...  No booking ID exists, so identical rows cannot...
1                     Bookings with no guests removed  ...             A booking requires at least one guest.
2            Zero-night bookings retained and flagged  ...  Some may be day-use records or data errors; th...
3                           Negative ADR rows removed  ...  A negative quoted room price is not valid for ...
4             ADR above EUR 1000 retained and flagged  ...  The record may be valid but is influential for...
5                Adults above 10 retained and flagged  ...                    The row may be a group booking.
6                   Missing children filled with zero  ...  Four missing values have no other invalid gues...
7         Missing country filled with UNK and flagged  ...  The booking remains usable while missingness s...
8         

                                              problem  ...                                             reason
0           Exact duplicate rows retained and flagged  ...  No booking ID exists, so identical rows cannot...
1                     Bookings with no guests removed  ...             A booking requires at least one guest.
2            Zero-night bookings retained and flagged  ...  Some may be day-use records or data errors; th...
3                           Negative ADR rows removed  ...  A negative quoted room price is not valid for ...
4             ADR above EUR 1000 retained and flagged  ...  The record may be valid but is influential for...
5                Adults above 10 retained and flagged  ...                    The row may be a group booking.
6                   Missing children filled with zero  ...  Four missing values have no other invalid gues...
7         Missing country filled with UNK and flagged  ...  The booking remains usable while missingness s...
8         

## Data limitations

- The source covers two Portuguese hotels from July 2015 through August 2017.
- Historical room inventory is absent, so occupancy and fully-booked dates cannot be calculated.
- The source has no booking ID and no customer ID.
- ADR excludes extras, food and beverage, taxes and refunds.
- The data does not show whether a cancelled room was resold.
